In [ ]:
from dotenv import load_dotenv
load_dotenv()

import os
import requests
import pandas as pd

pd.set_option('display.max_colwidth', None)

pd.set_option('display.max_columns', None)

gameName = ''
tagLine = ''
api_key = os.environ.get('riot_api_key')

def get_puuid(gameName=None, tagLine=None, api_key=None):
    link = f'https://americas.api.riotgames.com/riot/account/v1/accounts/by-riot-id/{gameName}/{tagLine}?api_key={api_key}'

    response = requests.get(link)
    return response.json()['puuid']

def get_name_tag(puuid=None, api_key=None):
    link = f'https://americas.api.riotgames.com/riot/account/v1/accounts/by-puuid/{puuid}?api_key={api_key}'

    response = requests.get(link)
    GameName = response.json()['gameName']
    TagLine = response.json()['tagLine']

    return f'{GameName}#{TagLine}'

def get_ladder(top=None):
    root = 'https://na1.api.riotgames.com/'
    chall = 'lol/league/v4/challengerleagues/by-queue/RANKED_SOLO_5x5'
    gm = 'lol/league/v4/grandmasterleagues/by-queue/RANKED_SOLO_5x5'
    master = 'lol/league/v4/masterleagues/by-queue/RANKED_SOLO_5x5'

    chall_response = requests.get(root + chall + '?api_key=' + api_key)
    gm_response = requests.get(root + gm + '?api_key=' + api_key)
    master_response = requests.get(root + master + '?api_key=' + api_key)

    chall_df = pd.DataFrame(chall_response.json()['entries']).sort_values('leaguePoints', ascending=False).reset_index(drop=True)
    gm_df = pd.DataFrame(gm_response.json()['entries']).sort_values('leaguePoints', ascending=False).reset_index(drop=True)
    master_df = pd.DataFrame(master_response.json()['entries']).sort_values('leaguePoints', ascending=False).reset_index(drop=True)

    ladder = pd.concat([chall_df, gm_df, master_df])[:top].reset_index(drop=True)
    ladder = ladder.drop(columns = 'rank').reset_index(drop=False).rename(columns={'index': 'rank'})
    ladder['rank'] = ladder['rank'] + 1

    return ladder

def get_match_history(region=None, puuid=None, start=0, count=20):
    root_url = f'https://{region}.api.riotgames.com/'
    endpoint = f'lol/match/v5/matches/by-puuid/{puuid}/ids'
    query_params = f'?start={start}&count={count}'

    response = requests.get(root_url + endpoint + query_params + '&api_key=' + api_key)
    
    return response.json()

def get_match_data_from_id(region=None, match_id=None):
    root_url = f'https://{region}.api.riotgames.com/'
    endpoint = f'lol/match/v5/matches/{match_id}'

    response = requests.get(root_url + endpoint + '?api_key=' + api_key)
    
    return response.json()

In [4]:
get_ladder(top=3000)

,rank,puuid,leaguePoints,wins,losses,veteran,inactive,freshBlood,hotStreak
0,1,TAKCv8Bg43_R7se1806uFc5MW_ZH9tcjAbxkx9HCrjwfD4zX9QBM_zv6oQ9gcJEENl1rr-v4e84Bkg,855,104,24,False,False,True,False
1,2,Fr13MrbTXNdr55Zy96MCHRvqPUvWVUV0w4DSvE__A2lZNXl1UYL_jECYNP9KD_IzuFh0q79hJbVwfQ,824,148,65,False,False,True,True
2,3,kqpJK0Zp7-t1NUrp-CNwkdzAC_Plk1qq-Uc8UtRCZcSllb3GBrsFAP9D0bSj0eFhRouoTB_kcD2SzQ,810,402,311,False,False,True,False
3,4,YLgM0EbM3emykEeKKQrhcne3DcNjhPmulIJBNE5cyJ989VzwtpAqgTf3EkqgpuYjki4jMQwg_LxWgw,546,216,165,True,False,False,False
4,5,BuQerXTF5J8PfS6QbPWKbpct97VT6Z_69_a0qCJD5vwucEBrMbgb7IttuZxcmGrnyCucMk9_E9diNw,480,184,140,False,False,False,True
...,...,...,...,...,...,...,...,...,...
2995,2996,yTHQd0Mr3j-6AIu49NovRSauC1CMReh4RnGIAvO2-4P3kX8xZBs4IlUFI0T1_d2H7Dkv1ISiNOAwXg,60,310,290,True,False,False,False
2996,2997,MSN_TrJ4moQD2haTLQdUW3UXoRbPcslOSMjTgQ9Qlm--l6DYgzD0Sn3UNV5V0GxxK4ZWP4ugZFt9HA,60,308,307,True,False,False,False
2997,2998,S56vMNVKhw97lno4b_B2lESeC2FeiuI7WDflwO2UtD268kfqdFbfMMm7lHHKEObfpSx6g5jqhzrCdQ,60,306,283,False,False,False,False
2998,2999,_sSrXk8uo7pMbyDpfrPMMzWtMbwDe3fayzyNnvXdkSwMCe6G-2MaPfC3T1sq31Y4OX7E_18yqRE9Rw,60,304,287,False,False,True,False


In [5]:
def process_match_json(match_json, puuid):
    metadata = match_json['metadata']
    info = match_json['info']
    players = info['participants']

    match_id = metadata['matchId']
    participants = metadata['participants']
    teams = info['teams']


    player = players[participants.index(puuid)]


    # -------------------------
    # Match information
    # -------------------------

    game_creation = info['gameCreation']
    game_duration = info['gameDuration']
    game_start_timestamp = info['gameStartTimestamp']
    game_end_timestamp = info['gameEndTimestamp']
    patch = info['gameVersion']


    # -------------------------
    # Player identity
    # -------------------------

    riot_id = player['riotIdGameName']
    riot_tag = player['riotIdTagline']
    summoner_id = player['summonerId']
    summoner_name = player['summonerName']


    # -------------------------
    # Target variable
    # -------------------------

    win = player['win']


    # -------------------------
    # Team identity
    # Riot uses teamId 100 and 200 instead of "team 1" and "team 2".
    # 100 is usually blue side, 200 is usually red side.
    # -------------------------

    team_id = player['teamId']

    if team_id == 100:
        enemy_team_id = 200
        side = 'blue'
    else:
        enemy_team_id = 100
        side = 'red'


    # -------------------------
    # Champion information
    # -------------------------

    champion_id = player['championId']
    champion_name = player['championName']
    champion_transform = player['championTransform']


    # -------------------------
    # Gold and farming
    # -------------------------

    gold_earned = player['goldEarned']
    neutral_minions_killed = player['neutralMinionsKilled']
    total_minions_killed = player['totalMinionsKilled']
    cs_total = neutral_minions_killed + total_minions_killed
    cs_per_min = cs_total / (game_duration / 60)


    # -------------------------
    # KDA
    # -------------------------

    kills = player['kills']
    deaths = player['deaths']
    assists = player['assists']
    firstblood = player['firstBloodKill']
    kda = (kills + assists) / max(1, deaths)


    # -------------------------
    # Damage
    # -------------------------

    total_damage_dealt = player['totalDamageDealt']
    total_damage_dealt_to_champions = player['totalDamageDealtToChampions']
    total_damage_shielded = player['totalDamageShieldedOnTeammates']
    total_damage_taken = player['totalDamageTaken']
    total_heal = player['totalHeal']
    total_time_cc_dealt = player['totalTimeCCDealt']


    # -------------------------
    # Surrender
    # -------------------------

    early_surrender = player['gameEndedInEarlySurrender']
    surrender = player['gameEndedInSurrender']


    # -------------------------
    # Items
    # -------------------------

    item0 = player['item0']
    item1 = player['item1']
    item2 = player['item2']
    item3 = player['item3']
    item4 = player['item4']
    item5 = player['item5']
    item6 = player['item6']


    # -------------------------
    # Objectives stolen
    # -------------------------

    objectives_stolen = player['objectivesStolen']
    objectives_stolen_assists = player['objectivesStolenAssists']


    # -------------------------
    # Summoner spells
    # -------------------------

    summoner1_id = player['summoner1Id']
    summoner2_id = player['summoner2Id']


    # -------------------------
    # Vision
    # -------------------------

    wards_placed = player['wardsPlaced']
    wards_killed = player['wardsKilled']
    vision_score = player['visionScore']
    detector_wards_placed = player['detectorWardsPlaced']


    # -------------------------
    # Role / position
    # -------------------------

    role = player['role']
    lane = player['lane']
    team_position = player['teamPosition']



    # -------------------------
    # Team-level first blood
    # If any player on your team has firstBloodKill == True,
    # then your team got first blood.
    # -------------------------

    team_first_blood = False
    enemy_first_blood = False
    first_blood_team_id = None
    first_blood_champion = None

    for p in players:
        if p['firstBloodKill'] == True:
            first_blood_team_id = p['teamId']
            first_blood_champion = p['championName']

            if p['teamId'] == team_id:
                team_first_blood = True
            else:
                enemy_first_blood = True


    # -------------------------
    # Team totals
    # These compare your player's team against the enemy team.
    # Useful for simple win/loss modeling.
    # -------------------------

    team_kills = 0
    team_deaths = 0
    team_assists = 0
    team_gold = 0
    team_damage_to_champions = 0
    team_vision_score = 0

    enemy_kills = 0
    enemy_deaths = 0
    enemy_assists = 0
    enemy_gold = 0
    enemy_damage_to_champions = 0
    enemy_vision_score = 0

    for p in players:
        if p['teamId'] == team_id:
            team_kills = team_kills + p['kills']
            team_deaths = team_deaths + p['deaths']
            team_assists = team_assists + p['assists']
            team_gold = team_gold + p['goldEarned']
            team_damage_to_champions = team_damage_to_champions + p['totalDamageDealtToChampions']
            team_vision_score = team_vision_score + p['visionScore']
        else:
            enemy_kills = enemy_kills + p['kills']
            enemy_deaths = enemy_deaths + p['deaths']
            enemy_assists = enemy_assists + p['assists']
            enemy_gold = enemy_gold + p['goldEarned']
            enemy_damage_to_champions = enemy_damage_to_champions + p['totalDamageDealtToChampions']
            enemy_vision_score = enemy_vision_score + p['visionScore']


    kill_difference = team_kills - enemy_kills
    gold_difference = team_gold - enemy_gold
    damage_difference = team_damage_to_champions - enemy_damage_to_champions
    vision_difference = team_vision_score - enemy_vision_score



    # -------------------------
    # Team objectives
    # -------------------------

    for team in teams:

        if team['teamId'] == team_id:
            team_objs = team['objectives']
        else:
            enemy_objs = team['objectives']


    barons = team_objs['baron']
    dragons = team_objs['dragon']
    grubs = team_objs.get('horde', {'first': False, 'kills': 0})
    heralds = team_objs['riftHerald']
    inhibitors = team_objs['inhibitor']
    towers = team_objs['tower']

    enemy_barons = enemy_objs['baron']
    enemy_dragons = enemy_objs['dragon']
    enemy_grubs = enemy_objs.get('horde', {'first': False, 'kills': 0})
    enemy_heralds = enemy_objs['riftHerald']
    enemy_inhibitors = enemy_objs['inhibitor']
    enemy_towers = enemy_objs['tower']


    baron_first = barons['first']
    baron_kills = barons['kills']

    dragon_first = dragons['first']
    dragon_kills = dragons['kills']

    grub_first = grubs['first']
    grub_kills = grubs['kills']

    herald_first = heralds['first']
    herald_kills = heralds['kills']

    inhibitor_first = inhibitors['first']
    inhibitor_kills = inhibitors['kills']

    tower_first = towers['first']
    tower_kills = towers['kills']


    enemy_baron_first = enemy_barons['first']
    enemy_baron_kills = enemy_barons['kills']

    enemy_dragon_first = enemy_dragons['first']
    enemy_dragon_kills = enemy_dragons['kills']

    enemy_grub_first = enemy_grubs['first']
    enemy_grub_kills = enemy_grubs['kills']

    enemy_herald_first = enemy_heralds['first']
    enemy_herald_kills = enemy_heralds['kills']

    enemy_inhibitor_first = enemy_inhibitors['first']
    enemy_inhibitor_kills = enemy_inhibitors['kills']

    enemy_tower_first = enemy_towers['first']
    enemy_tower_kills = enemy_towers['kills']


    dragon_difference = dragon_kills - enemy_dragon_kills
    baron_difference = baron_kills - enemy_baron_kills
    grub_difference = grub_kills - enemy_grub_kills
    herald_difference = herald_kills - enemy_herald_kills
    inhibitor_difference = inhibitor_kills - enemy_inhibitor_kills
    tower_difference = tower_kills - enemy_tower_kills



    # -------------------------
    # Put everything into one row
    # -------------------------

    row = {
        'match_id': match_id,
        'game_creation': game_creation,
        'game_duration': game_duration,
        'game_start_timestamp': game_start_timestamp,
        'game_end_timestamp': game_end_timestamp,
        'patch': patch,

        'puuid': puuid,
        'riot_id': riot_id,
        'riot_tag': riot_tag,
        'summoner_id': summoner_id,
        'summoner_name': summoner_name,

        'win': win,

        'team_id': team_id,
        'enemy_team_id': enemy_team_id,
        'side': side,

        'champion_id': champion_id,
        'champion_name': champion_name,
        'champion_transform': champion_transform,

        'gold_earned': gold_earned,
        'neutral_minions_killed': neutral_minions_killed,
        'total_minions_killed': total_minions_killed,
        'cs_total': cs_total,
        'cs_per_min': cs_per_min,

        'kills': kills,
        'deaths': deaths,
        'assists': assists,
        'firstblood': firstblood,
        'kda': kda,

        'total_damage_dealt': total_damage_dealt,
        'total_damage_dealt_to_champions': total_damage_dealt_to_champions,
        'total_damage_shielded': total_damage_shielded,
        'total_damage_taken': total_damage_taken,
        'total_heal': total_heal,
        'total_time_cc_dealt': total_time_cc_dealt,

        'early_surrender': early_surrender,
        'surrender': surrender,

        'item0': item0,
        'item1': item1,
        'item2': item2,
        'item3': item3,
        'item4': item4,
        'item5': item5,
        'item6': item6,

        'objectives_stolen': objectives_stolen,
        'objectives_stolen_assists': objectives_stolen_assists,

        'summoner1_id': summoner1_id,
        'summoner2_id': summoner2_id,

        'wards_placed': wards_placed,
        'wards_killed': wards_killed,
        'vision_score': vision_score,
        'detector_wards_placed': detector_wards_placed,

        'role': role,
        'lane': lane,
        'team_position': team_position,


        'team_first_blood': team_first_blood,
        'enemy_first_blood': enemy_first_blood,
        'first_blood_team_id': first_blood_team_id,
        'first_blood_champion': first_blood_champion,

        'team_kills': team_kills,
        'team_deaths': team_deaths,
        'team_assists': team_assists,
        'team_gold': team_gold,
        'team_damage_to_champions': team_damage_to_champions,
        'team_vision_score': team_vision_score,

        'enemy_kills': enemy_kills,
        'enemy_deaths': enemy_deaths,
        'enemy_assists': enemy_assists,
        'enemy_gold': enemy_gold,
        'enemy_damage_to_champions': enemy_damage_to_champions,
        'enemy_vision_score': enemy_vision_score,

        'kill_difference': kill_difference,
        'gold_difference': gold_difference,
        'damage_difference': damage_difference,
        'vision_difference': vision_difference,


        'baron_first': baron_first,
        'baron_kills': baron_kills,

        'dragon_first': dragon_first,
        'dragon_kills': dragon_kills,

        'grub_first': grub_first,
        'grub_kills': grub_kills,

        'herald_first': herald_first,
        'herald_kills': herald_kills,

        'inhibitor_first': inhibitor_first,
        'inhibitor_kills': inhibitor_kills,

        'tower_first': tower_first,
        'tower_kills': tower_kills,


        'enemy_baron_first': enemy_baron_first,
        'enemy_baron_kills': enemy_baron_kills,

        'enemy_dragon_first': enemy_dragon_first,
        'enemy_dragon_kills': enemy_dragon_kills,

        'enemy_grub_first': enemy_grub_first,
        'enemy_grub_kills': enemy_grub_kills,

        'enemy_herald_first': enemy_herald_first,
        'enemy_herald_kills': enemy_herald_kills,

        'enemy_inhibitor_first': enemy_inhibitor_first,
        'enemy_inhibitor_kills': enemy_inhibitor_kills,

        'enemy_tower_first': enemy_tower_first,
        'enemy_tower_kills': enemy_tower_kills,


        'dragon_difference': dragon_difference,
        'baron_difference': baron_difference,
        'grub_difference': grub_difference,
        'herald_difference': herald_difference,
        'inhibitor_difference': inhibitor_difference,
        'tower_difference': tower_difference
    }


    # -------------------------
    # Convert one row into a DataFrame
    # -------------------------

    matchdf = pd.DataFrame([row])
    return matchdf


In [ ]:
puuid = ''

In [868]:
match_ids = get_match_history(region='americas', puuid=puuid, start=0, count=40)

In [869]:
df = pd.DataFrame()

for match_id in match_ids:
    game = get_match_data_from_id(region='americas', match_id=match_id)
    matchDF = process_match_json(game, puuid=puuid)

    df = pd.concat([df, matchDF], ignore_index=True)

In [870]:
df

,match_id,game_creation,game_duration,game_start_timestamp,game_end_timestamp,patch,puuid,riot_id,riot_tag,summoner_id,summoner_name,win,team_id,enemy_team_id,side,champion_id,champion_name,champion_transform,gold_earned,neutral_minions_killed,total_minions_killed,cs_total,cs_per_min,kills,deaths,assists,firstblood,kda,total_damage_dealt,total_damage_dealt_to_champions,total_damage_shielded,total_damage_taken,total_heal,total_time_cc_dealt,early_surrender,surrender,item0,item1,item2,item3,item4,item5,item6,objectives_stolen,objectives_stolen_assists,summoner1_id,summoner2_id,wards_placed,wards_killed,vision_score,detector_wards_placed,role,lane,team_position,team_first_blood,enemy_first_blood,first_blood_team_id,first_blood_champion,team_kills,team_deaths,team_assists,team_gold,team_damage_to_champions,team_vision_score,enemy_kills,enemy_deaths,enemy_assists,enemy_gold,enemy_damage_to_champions,enemy_vision_score,kill_difference,gold_difference,damage_difference,vision_difference,baron_first,baron_kills,dragon_first,dragon_kills,grub_first,grub_kills,herald_first,herald_kills,inhibitor_first,inhibitor_kills,tower_first,tower_kills,enemy_baron_first,enemy_baron_kills,enemy_dragon_first,enemy_dragon_kills,enemy_grub_first,enemy_grub_kills,enemy_herald_first,enemy_herald_kills,enemy_inhibitor_first,enemy_inhibitor_kills,enemy_tower_first,enemy_tower_kills,dragon_difference,baron_difference,grub_difference,herald_difference,inhibitor_difference,tower_difference
0,NA1_5552207012,1777768479842,1397,1777768493326,1777769890811,16.9.772.1032,BMi4CNpKIhEFggSmMTqSox661sMW2DGvueZ8O_jqJ_zM29GfX_8vbw5q-SacpCxpIDwlxFSBw_Iqew,KataIina,NA1,uu7a_UQZ215o3IvNhYwqRwGyta6SwITnBYaz-WzJCOMzpLAJ,,True,200,100,red,134,Syndra,0,10651,0,189,189,8.117394,4,4,10,False,3.500000,136392,18478,0,16550,2210,490,False,False,2031,2503,1056,3173,4645,0,3363,0,0,12,4,8,0,15,0,SOLO,MIDDLE,MIDDLE,False,True,100,Aatrox,34,23,42,55656,97607,116,23,35,31,45026,79342,91,11,10630,18265,25,True,1,True,3,True,3,False,0,True,2,False,9,False,0,False,0,False,0,True,1,False,0,True,3,3,1,3,-1,2,6
1,NA1_5552174640,1777766039424,1963,1777766061627,1777768024695,16.9.772.1032,BMi4CNpKIhEFggSmMTqSox661sMW2DGvueZ8O_jqJ_zM29GfX_8vbw5q-SacpCxpIDwlxFSBw_Iqew,KataIina,NA1,uu7a_UQZ215o3IvNhYwqRwGyta6SwITnBYaz-WzJCOMzpLAJ,,False,100,200,blue,799,Ambessa,0,14653,201,36,237,7.244014,8,9,4,False,1.333333,323024,22989,0,37025,13622,423,False,False,6698,6610,1036,6694,6333,3047,3364,0,0,11,4,12,4,35,3,NONE,JUNGLE,JUNGLE,True,False,100,Anivia,38,43,34,66321,134586,219,43,38,76,74915,129806,220,-5,-8594,4780,-1,True,1,False,1,True,3,True,1,False,0,True,4,False,0,True,4,False,0,False,0,True,3,False,11,-3,1,3,1,-3,-7
2,NA1_5552145422,1777763384816,2192,1777763402998,1777765594625,16.9.772.1032,BMi4CNpKIhEFggSmMTqSox661sMW2DGvueZ8O_jqJ_zM29GfX_8vbw5q-SacpCxpIDwlxFSBw_Iqew,KataIina,NA1,uu7a_UQZ215o3IvNhYwqRwGyta6SwITnBYaz-WzJCOMzpLAJ,,False,100,200,blue,799,Ambessa,0,15056,210,59,269,7.363139,4,9,7,False,1.222222,348470,24777,0,44525,16441,419,False,False,6698,6610,3156,3111,1036,6333,3364,0,0,11,4,3,3,29,2,NONE,JUNGLE,JUNGLE,False,True,200,Zed,28,43,41,70738,128137,251,43,28,71,79266,184677,217,-15,-8528,-56540,34,False,0,True,1,True,3,False,0,False,0,False,4,True,1,False,4,False,0,True,1,True,1,True,8,-3,-1,3,-1,-1,-4
3,NA1_5552124829,1777761201033,1921,1777761219067,1777763140324,16.9.772.1032,BMi4CNpKIhEFggSmMTqSox661sMW2DGvueZ8O_jqJ_zM29GfX_8vbw5q-SacpCxpIDwlxFSBw_Iqew,KataIina,NA1,uu7a_UQZ215o3IvNhYwqRwGyta6SwITnBYaz-WzJCOMzpLAJ,,True,200,100,red,876,Lillia,0,18780,224,53,277,8.651744,15,5,16,False,6.200000,392004,44491,0,50301,32221,551,False,False,3157,6653,4633,3041,3047,3089,3364,0,0,11,4,6,1,20,0,NONE,JUNGLE,JUNGLE,False,True,100,Ezreal,40,33,59,76010,139459,191,33,40,32,65996,136116,232,7,10014,3343,-41,False,0,False,2,False,0,False,0,True,3,False,11,True,1,True,2,True,3,False,0,False,0,True,6,0,-1,-3,0,3,5
4,NA1_5551721292,1777699609356,1574,1777699629509,1777701203202,16.9.772.1032,BMi

In [871]:
import os
from dotenv import load_dotenv
load_dotenv()

from sqlalchemy import create_engine
from pangres import upsert

db_username = os.environ.get('dbusername')
db_password = os.environ.get('dbpassword')
db_host = os.environ.get('db_host')
db_port = os.environ.get('db_port')
db_name = os.environ.get('db_name')

def create_db_connection_url(db_username, db_password, db_host, db_port, db_name):
    connection_url = 'postgresql+psycopg2://' + db_username + ':' + db_password + '@' + db_host + ':' + db_port + '/' + db_name
    return connection_url

con_url = create_db_connection_url(db_username, db_password, db_host, db_port, db_name)

db_engine = create_engine(con_url, pool_recycle=3600)

connection = db_engine.connect()

In [872]:
df['uuid'] = df['match_id'] + '_' + df['puuid']
df = df.set_index('uuid')

In [873]:
df

,match_id,game_creation,game_duration,game_start_timestamp,game_end_timestamp,patch,puuid,riot_id,riot_tag,summoner_id,summoner_name,win,team_id,enemy_team_id,side,champion_id,champion_name,champion_transform,gold_earned,neutral_minions_killed,total_minions_killed,cs_total,cs_per_min,kills,deaths,assists,firstblood,kda,total_damage_dealt,total_damage_dealt_to_champions,total_damage_shielded,total_damage_taken,total_heal,total_time_cc_dealt,early_surrender,surrender,item0,item1,item2,item3,item4,item5,item6,objectives_stolen,objectives_stolen_assists,summoner1_id,summoner2_id,wards_placed,wards_killed,vision_score,detector_wards_placed,role,lane,team_position,team_first_blood,enemy_first_blood,first_blood_team_id,first_blood_champion,team_kills,team_deaths,team_assists,team_gold,team_damage_to_champions,team_vision_score,enemy_kills,enemy_deaths,enemy_assists,enemy_gold,enemy_damage_to_champions,enemy_vision_score,kill_difference,gold_difference,damage_difference,vision_difference,baron_first,baron_kills,dragon_first,dragon_kills,grub_first,grub_kills,herald_first,herald_kills,inhibitor_first,inhibitor_kills,tower_first,tower_kills,enemy_baron_first,enemy_baron_kills,enemy_dragon_first,enemy_dragon_kills,enemy_grub_first,enemy_grub_kills,enemy_herald_first,enemy_herald_kills,enemy_inhibitor_first,enemy_inhibitor_kills,enemy_tower_first,enemy_tower_kills,dragon_difference,baron_difference,grub_difference,herald_difference,inhibitor_difference,tower_difference
uuid,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
NA1_5552207012_BMi4CNpKIhEFggSmMTqSox661sMW2DGvueZ8O_jqJ_zM29GfX_8vbw5q-SacpCxpIDwlxFSBw_Iqew,NA1_5552207012,1777768479842,1397,1777768493326,1777769890811,16.9.772.1032,BMi4CNpKIhEFggSmMTqSox661sMW2DGvueZ8O_jqJ_zM29GfX_8vbw5q-SacpCxpIDwlxFSBw_Iqew,KataIina,NA1,uu7a_UQZ215o3IvNhYwqRwGyta6SwITnBYaz-WzJCOMzpLAJ,,True,200,100,red,134,Syndra,0,10651,0,189,189,8.117394,4,4,10,False,3.500000,136392,18478,0,16550,2210,490,False,False,2031,2503,1056,3173,4645,0,3363,0,0,12,4,8,0,15,0,SOLO,MIDDLE,MIDDLE,False,True,100,Aatrox,34,23,42,55656,97607,116,23,35,31,45026,79342,91,11,10630,18265,25,True,1,True,3,True,3,False,0,True,2,False,9,False,0,False,0,False,0,True,1,False,0,True,3,3,1,3,-1,2,6
NA1_5552174640_BMi4CNpKIhEFggSmMTqSox661sMW2DGvueZ8O_jqJ_zM29GfX_8vbw5q-SacpCxpIDwlxFSBw_Iqew,NA1_5552174640,1777766039424,1963,1777766061627,1777768024695,16.9.772.1032,BMi4CNpKIhEFggSmMTqSox661sMW2DGvueZ8O_jqJ_zM29GfX_8vbw5q-SacpCxpIDwlxFSBw_Iqew,KataIina,NA1,uu7a_UQZ215o3IvNhYwqRwGyta6SwITnBYaz-WzJCOMzpLAJ,,False,100,200,blue,799,Ambessa,0,14653,201,36,237,7.244014,8,9,4,False,1.333333,323024,22989,0,37025,13622,423,False,False,6698,6610,1036,6694,6333,3047,3364,0,0,11,4,12,4,35,3,NONE,JUNGLE,JUNGLE,True,False,100,Anivia,38,43,34,66321,134586,219,43,38,76,74915,129806,220,-5,-8594,4780,-1,True,1,False,1,True,3,True,1,False,0,True,4,False,0,True,4,False,0,False,0,True,3,False,11,-3,1,3,1,-3,-7
NA1_5552145422_BMi4CNpKIhEFggSmMTqSox661sMW2DGvueZ8O_jqJ_zM29GfX_8vbw5q-SacpCxpIDwlxFSBw_Iqew,NA1_5552145422,1777763384816,2192,1777763402998,1777765594625,16.9.772.1032,BMi4CNpKIhEFggSmMTqSox661sMW2DGvueZ8O_jqJ_zM29GfX_8vbw5q-SacpCxpIDwlxFSBw_Iqew,KataIina,NA1,uu7a_UQZ215o3IvNhYwqRwGyta6SwITnBYaz-WzJCOMzpLAJ,,False,100,200,blue,799,Ambessa,0,15056,210,59,269,7.363139,4,9,7,False,1.222222,348470,24777,0,44525,16441,419,False,False,6698,6610,3156,3111,1036,6333,3364,0,0,11,4,3,3,29,2,NONE,JUNGLE,JUNGLE,False,True,200,Zed,28,43,41,70738,128137,251,43,28,71,79266,184677,217,-15,-8528,-56540,34,False,0,True,1,True,3,False,0,False,0,False,4,True,1,False,4,False,0,True,1,True,1,True,8,-3,-1,3,-1,-1,-4
NA1_5552124829_BMi4CNpKIhEFggSmMTqSox661sMW2DGvueZ8O_jqJ_zM29GfX_8vbw5q-SacpCxpIDwlxFSBw_Iqew,NA1_5552124829,1777761201033,1921,1777761219067,1777763140324,16.9.772.1032,BMi4CNpKIhEFggSmMTqSox661sMW2DGvueZ8O_jqJ_zM29GfX_8vbw5q-SacpCxpIDwlxFSBw_Iqew,KataIina,NA1,uu7a_UQZ215o3IvNhYwqRwGyta6SwITnBYaz-WzJCOMzp

In [874]:
upsert(con=connection, df=df, schema='soloq', table_name='match_data', create_table=True, create_schema=True, if_row_exists='update')

In [875]:
connection.commit()